# Lesson 1 — reference solutions

*Do not show the student* — for assessor use only.

## Exercise 1

- a) **Convex NLP / SDP** — minimizing an L2 norm subject to an LMI is a convex SDP (semidefinite). Reformulable as SOCP if the LMI is simpler.
- b) **MILP** — discrete shift assignments → integer/binary variables; objective and constraints linear.
- c) **NLP/MINLP** — DAE discretization yields a large NLP; *if* there are integer design variables (number of stages, equipment selection) it becomes an MINLP.
- d) **MINLP, nonconvex** — pairwise distance constraints are nonconvex (the feasible set $\|x_i - x_j\| \ge d$ is the *complement* of a ball).

In [ ]:
import numpy as np
import discopt as do

c = np.array([0.50, 0.80, 0.30, 0.40])
A = np.array([[250, 120, 350, 150], [8, 8, 20, 12], [30, 300, 60, 50]])
b = np.array([2000, 50, 800])

m = do.Model("diet_extra")
x = m.add_variables(4, lb=0, ub=10, name="x")
z = m.add_variables(4, vtype="binary", name="z")
M = 10  # big-M
m.add_constraints(A @ x >= b)
for i in range(4):
    m.add_constraint(x[i] <= M * z[i])
m.add_constraint(z.sum() >= 2)
m.minimize(c @ x)
res = m.solve()
print("foods:", res.x, "cost:", res.objective)


In [ ]:
rng = np.random.default_rng(0)
n = 10
v = rng.integers(1, 20, size=n); w = rng.integers(1, 15, size=n); W = int(0.4 * w.sum())

def solve(binary):
    m = do.Model("k")
    x = m.add_variables(n, vtype="binary", name="x") if binary else m.add_variables(n, lb=0, ub=1, name="x")
    m.add_constraint(w @ x <= W); m.maximize(v @ x)
    return m.solve()

zlp = solve(False).objective; zip_ = solve(True).objective
print(f"LP={zlp:.3f}  MILP={zip_}  gap={(zlp - zip_)/zip_:.3%}")


In [ ]:
from discopt.modeling import Model
seen = []
for x10 in np.linspace(-2, 2, 5):
    for x20 in np.linspace(-1.5, 1.5, 5):
        m = Model("c")
        x1 = m.add_variable(lb=-3, ub=3); x2 = m.add_variable(lb=-2, ub=2)
        f = (4 - 2.1*x1**2 + x1**4/3)*x1**2 + x1*x2 + (-4 + 4*x2**2)*x2**2
        m.minimize(f)
        r = m.solve(mode="local", x0=[x10, x20])
        seen.append(round(r.objective, 3))
print("unique objectives:", sorted(set(seen)))


In [ ]:
m = do.Model("diet")
c = np.array([0.50, 0.80, 0.30])
A = np.array([[250, 120, 350], [8, 8, 20], [30, 300, 60]])
b = np.array([2000, 50, 800])
x = m.add_variables(3, lb=0); m.add_constraints(A @ x >= b); m.minimize(c @ x)
m.solve(verbose=True)


**Solution explanation:**

- **Primal residual** $\|Ax - b\|$ — how far the current iterate is from satisfying the constraints (feasibility gap).
- **Dual residual** $\|\nabla_x L(x, \lambda)\|$ — how far we are from the stationarity condition of the Lagrangian (optimality gap on the dual side).
- **Complementary slackness** $|\lambda_i (a_i^\top x - b_i)|$ — multiplier × constraint slack must vanish at optimum (active multiplier ↔ active constraint).